# Stage 4: Data Validation & Cleaning
Capstone Project: Spam Message Analysis Using Malicious URLs

Remove duplicate URLs, validate the extracted features, flag length outliers, and re-run the data-quality audit to prove the problems were fixed.

In [1]:
from pathlib import Path

# Project root = parent of this notebook's stage folder (run the notebook from inside its own folder)
ROOT = Path.cwd().parent
RAW_CSV = ROOT / "dataset" / "dataset.csv"
INTER = ROOT / "dataset" / "intermediate"      # output of each stage becomes input of the next
RESULTS = ROOT / "results"
GRAPH_DIR = RESULTS / "graphs"
for p in (INTER, RESULTS, GRAPH_DIR):
    p.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
import numpy as np

df_final = pd.read_csv(INTER / "stage3_features.csv", keep_default_na=False)
print("Loaded from Stage 3:", df_final.shape)


Loaded from Stage 3: (651066, 19)


### Data Quality Audit — BEFORE Cleaning
Quantify every known issue in the raw file before touching it, so we have real before/after numbers for the review.

In [3]:
def data_quality_audit(data, label):
    print(f"===== DATA QUALITY AUDIT: {label} =====")
    print("Total rows:", len(data))
    print("Missing url:", data["url"].isna().sum())
    print("Missing type:", data["type"].isna().sum())
    print("Duplicate urls:", data["url"].duplicated().sum())

    non_null_url = data["url"].dropna().astype(str)
    print("URLs with leading/trailing whitespace:", (non_null_url != non_null_url.str.strip()).sum())

    non_null_type = data["type"].dropna().astype(str)
    print("Type values not already lowercase:", (non_null_type != non_null_type.str.lower()).sum())

    valid_lower = {"benign", "phishing", "defacement", "malware"}
    print("Invalid/garbage type labels:", (~non_null_type.str.strip().str.lower().isin(valid_lower)).sum())

    print("Empty-string or whitespace-only urls:", (non_null_url.str.strip().str.len() == 0).sum())
    print("Extremely long urls (>1000 chars):", (non_null_url.str.len() > 1000).sum())

    zero_width_pattern = r"[\u200b\ufeff]"
    print("URLs containing zero-width/BOM artifacts:", non_null_url.str.contains(zero_width_pattern, regex=True, na=False).sum())
    print()


## Stage 4 — Data Validation & Cleaning
Remove duplicate URLs and check the quality of the features we just extracted.

In [4]:
print("Duplicate URLs before cleaning:", df_final["url"].duplicated().sum())

df_final = df_final.drop_duplicates(subset=["url"]).reset_index(drop=True)

print("Row count after removing duplicates:", len(df_final))
print("\nMissing values per column:")
print(df_final.isna().sum()[df_final.isna().sum() > 0])

Duplicate URLs before cleaning: 10978
Row count after removing duplicates: 640088

Missing values per column:
Series([], dtype: int64)


In [5]:
# Validate the extraction step itself: how many URLs failed to resolve a domain?
failed_domain = (df_final["domain"].str.strip(".") == "").sum()
print("URLs with no resolvable domain:", failed_domain)

# Flag extreme outliers in url_length using the IQR rule (reported, not silently dropped)
q1, q3 = df_final["url_length"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
outlier_count = (df_final["url_length"] > upper_bound).sum()
print(f"URL length outliers (> {upper_bound:.0f} chars): {outlier_count} "
      f"({outlier_count/len(df_final)*100:.2f}% of data)")

print("\nFinal cleaned & validated dataset shape:", df_final.shape)

URLs with no resolvable domain: 21
URL length outliers (> 142 chars): 26473 (4.14% of data)

Final cleaned & validated dataset shape: (640088, 19)


### Data Quality Audit — AFTER Cleaning
Re-run the same audit on the cleaned data to prove each injected issue was actually resolved.

In [6]:
data_quality_audit(df_final, "CLEANED DATASET")

===== DATA QUALITY AUDIT: CLEANED DATASET =====
Total rows: 640088
Missing url: 0
Missing type: 0
Duplicate urls: 0
URLs with leading/trailing whitespace: 0
Type values not already lowercase: 0
Invalid/garbage type labels: 0
Empty-string or whitespace-only urls: 0
Extremely long urls (>1000 chars): 25
URLs containing zero-width/BOM artifacts: 0



In [7]:
df_final.to_csv(INTER / "stage4_clean.csv", index=False)
print("Saved:", INTER / "stage4_clean.csv")

Saved: C:\Users\karth\Desktop\spam-message-analysis-malicious-urls\dataset\intermediate\stage4_clean.csv
